# 03 · SVM, KNN y Naive Bayes: margen, vecindad y probabilidad

Tres familias clásicas que siguen siendo útiles y que enseñan ideas fundamentales: distancia, margen y modelamiento probabilístico.

## Objetivos
- Entender por qué KNN y SVM son sensibles a escala.
- Comparar kernels lineal y RBF.
- Interpretar `C` y `gamma` en SVM.
- Explorar la maldición de la dimensionalidad en KNN.
- Diferenciar Gaussian, Multinomial y Bernoulli Naive Bayes.
- Comparar accuracy, F1 y matrices de confusión.


## 1. K-Nearest Neighbors
KNN no aprende una ecuación explícita: conserva los datos y predice usando los vecinos más próximos.

**Ventajas:** intuitivo, no lineal, buen baseline local.
**Desventajas:** inferencia costosa, sensible a escala/ruido y empeora en alta dimensión.

La elección de distancia (Euclidiana, Manhattan, cosine) y `k` cambia el sesgo/varianza. K pequeño tiende a alta variance; K grande suaviza demasiado.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, classification_report
SEED=42
X,y=load_digits(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=SEED,stratify=y)

In [ ]:
rows=[]
for k in [1,3,5,9,15,25]:
    model=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=k,weights='distance'))
    cv=cross_val_score(model,Xtr,ytr,cv=4,scoring='accuracy')
    model.fit(Xtr,ytr); rows.append([k,cv.mean(),model.score(Xte,yte)])
pd.DataFrame(rows,columns=['k','cv_accuracy','test_accuracy']).round(4)

## 2. Support Vector Machines
Una SVM lineal busca el hiperplano con mayor margen. Los ejemplos que tocan el margen son **support vectors**.

- `C` alto penaliza fuerte errores: frontera más ajustada y potencialmente más variance.
- `C` bajo permite errores para un margen más amplio.
- Kernel RBF implementa una similitud no lineal. `gamma` controla qué tan local es la influencia de cada punto.

SVM funciona muy bien en datasets pequeños/medianos y espacios de alta dimensión, como texto con TF-IDF.


In [ ]:
models={
 'SVM linear':make_pipeline(StandardScaler(),SVC(kernel='linear',C=1)),
 'SVM RBF':make_pipeline(StandardScaler(),SVC(kernel='rbf',C=5,gamma='scale')),
 'KNN':make_pipeline(StandardScaler(),KNeighborsClassifier(5)),
 'GaussianNB':GaussianNB()
}
rows=[]
for name,m in models.items():
    m.fit(Xtr,ytr); p=m.predict(Xte)
    rows.append([name,accuracy_score(yte,p),f1_score(yte,p,average='macro')])
pd.DataFrame(rows,columns=['modelo','accuracy','macro_f1']).sort_values('macro_f1',ascending=False).round(4)

## 3. Visualizar fronteras no lineales
El dataset `make_moons` permite ver la diferencia entre un hiperplano lineal y un kernel RBF.


In [ ]:
Xm,ym=make_moons(n_samples=400,noise=.25,random_state=SEED)
def plot_boundary(model,ax,title):
    model.fit(Xm,ym)
    xx,yy=np.meshgrid(np.linspace(Xm[:,0].min()-.5,Xm[:,0].max()+.5,250),np.linspace(Xm[:,1].min()-.5,Xm[:,1].max()+.5,250))
    zz=model.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,zz,alpha=.25); ax.scatter(Xm[:,0],Xm[:,1],c=ym,s=15); ax.set_title(title)
fig,ax=plt.subplots(1,3,figsize=(15,4))
plot_boundary(make_pipeline(StandardScaler(),SVC(kernel='linear')),ax[0],'SVM linear')
plot_boundary(make_pipeline(StandardScaler(),SVC(kernel='rbf',gamma=2,C=3)),ax[1],'SVM RBF')
plot_boundary(make_pipeline(StandardScaler(),KNeighborsClassifier(15)),ax[2],'KNN')
plt.show()

## 4. Naive Bayes
Usa Bayes:
$$P(y|x)\propto P(y)P(x|y)$$
con la suposición 'naive' de independencia condicional entre features. Aunque esta suposición suele ser falsa, funciona sorprendentemente bien en muchos problemas.

- **GaussianNB:** features continuas aproximadamente gaussianas.
- **MultinomialNB:** conteos o TF-IDF no negativos; clásico en NLP.
- **BernoulliNB:** features binarias/presencia-ausencia.


In [ ]:
# Digits es no negativo, por lo que podemos comparar Gaussian y Multinomial NB
for m in [GaussianNB(),MultinomialNB(alpha=1.0),BernoulliNB(alpha=1.0,binarize=8)]:
    m.fit(Xtr,ytr); p=m.predict(Xte); print(type(m).__name__,round(accuracy_score(yte,p),4))

## 5. Maldición de dimensionalidad
Al crecer la dimensión, las distancias tienden a volverse menos discriminativas y los vecinos dejan de ser realmente 'cercanos'. Esto afecta especialmente a KNN y métodos basados en distancia. Reducción dimensional, selección de features y métricas de similitud adecuadas ayudan.

## Cuándo usar
- **KNN:** pequeños datasets, recomendación/similitud local, baseline no paramétrico.
- **SVM lineal:** texto sparse, alta dimensión y datasets medianos.
- **SVM RBF:** fronteras no lineales con pocos miles/decenas de miles de filas.
- **Naive Bayes:** texto, filtros, clasificación rápida y probabilística.

## Ejercicios
1. Mide tiempo de entrenamiento e inferencia de KNN vs SVM.
2. Haz una grilla de `C` y `gamma` y dibuja un heatmap del CV score.
3. Compara Euclidean vs Manhattan en KNN.
4. Aplica PCA antes de KNN y comprueba si mejora velocidad/score.
5. En un corpus de texto, compara MultinomialNB con LinearSVC usando TF-IDF.
6. Explica qué significa realmente un support vector.
